# GlassDesign ML — самостоятельный запуск (версия 409)

Это воспроизводимая демонстрация обратного проектирования составов оксидных стекол. Не требуется собственная папка Google Drive. Приложение загружает четыре обученные модели с опубликованных автором ссылок Google Drive и необходимые справочные данные из GitHub. Результаты носят расчётный характер и требуют экспериментального подтверждения.

**Инструкция:** «Среда выполнения → Выполнить все»; после запуска в последней ячейке появится временная ссылка Cloudflare на веб-интерфейс. Для работы приложения нужно сохранять сеанс Colab активным.

Первая загрузка моделей может быть длительной и потребует достаточно дисковой памяти/ОЗУ. Модели в формате joblib следует загружать только из доверенного источника.


In [ ]:
import os, sys, subprocess, pathlib, hashlib, gzip, shutil, json, requests

REPO='polinashidlovskaya/glassdesign-ml'
PROJECT=pathlib.Path('/content/glassdesign-ml')
if PROJECT.exists() and not (PROJECT/'.git').is_dir():
    raise RuntimeError('Уже существует папка проекта без Git. Создайте новую среду выполнения Colab.')
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1',f'https://github.com/{REPO}.git',str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'fetch','--depth','1','origin','main'],check=True)
    changed=subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if changed:
        raise RuntimeError('Локальный код изменён. Используйте новую среду Colab.')
    subprocess.run(['git','-C',str(PROJECT),'reset','--hard','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt'),'gdown'],check=True)

ROOT=PROJECT/'runtime'
os.environ['GLASSDESIGN_ROOT']=str(ROOT)
BASE_URL=f'https://raw.githubusercontent.com/{REPO}/main/public_data/'

def get_verified(url,dest,sha):
    dest=pathlib.Path(dest)
    if dest.exists() and hashlib.sha256(dest.read_bytes()).hexdigest()==sha:
        return dest
    dest.parent.mkdir(parents=True,exist_ok=True)
    tmp=dest.with_name(dest.name+'.downloading')
    try:
        with requests.get(url,stream=True,timeout=180) as response:
            response.raise_for_status()
            with tmp.open('wb') as out:
                for chunk in response.iter_content(chunk_size=1048576):
                    if chunk:out.write(chunk)
        observed=hashlib.sha256(tmp.read_bytes()).hexdigest()
        if observed!=sha:
            raise ValueError(f'Нарушена целостность {dest.name}: {observed} != {sha}')
        tmp.replace(dest)
    finally:
        tmp.unlink(missing_ok=True)
    return dest

# Четыре эталонные таблицы для поиска ближайших измеренных составов.
DATASETS={'density_dataset_clean.csv': ('Source_Data/density_dataset_clean.csv', '68e22bacbba9144b915ce74d763d34c6ca6200991ac44cd88b5238c7459d359c', '9ad4ffe16953f08863bfbafa3f3423ed1e43626d42f25900e8807b570491fcc2'), 'refractive_training_reconstructed.csv': ('GlassDesignML_Upgrade/refractive_training_reconstructed.csv', '6fbf8e30abe0aab31664832ef474a55230ba94b98e3074468534d83115d0c7a9', '68562f58a65e58863bb9ea17a3aedb038a3c7704d715429e104cc604f2fd7555'), 'tg_train_clean.csv': ('Tg/tg_train_clean.csv', 'a6ce60bb87490b015c45661d4f81e9c41623ffd125099a080ec133548ae9c757', '7ed0613159e6758becf8d45673b3727bb20f920f9d61739c57715a99076cca68'), 'softening_train_clean.csv': ('Softening/softening_train_clean.csv', '95a73795cf4f4f86e33bdb37c038d68dbd8128d1e7ea1f7a58ebdb4fbdb3b758', '5c2f1889e5a2cea25c4a04c8a8fe9aadfc968a466ea97f30eb3982e5c388223d')}
for filename,(relative,gz_hash,raw_hash) in DATASETS.items():
    dest=ROOT/relative
    if not dest.is_file() or hashlib.sha256(dest.read_bytes()).hexdigest()!=raw_hash:
        compressed=get_verified(BASE_URL+filename+'.gz',ROOT/'_downloads'/(filename+'.gz'),gz_hash)
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_name(dest.name+'.tmp')
        with gzip.open(compressed,'rb') as source, temp.open('wb') as target:
            shutil.copyfileobj(source,target)
        if hashlib.sha256(temp.read_bytes()).hexdigest()!=raw_hash:
            temp.unlink(missing_ok=True)
            raise ValueError(f'Контрольная сумма таблицы не совпала: {filename}')
        temp.replace(dest)
    print('✅ Таблица:',filename)

# Фиксированный SciGlass-индекс из трёх частей, собранный побайтно и проверенный.
index_dir=ROOT/'GlassDesignML_Upgrade'/'SciGlass_Universal_Literature_step398'/'cache'
index_file=index_dir/'sciglass_four_properties_mol_398_1.npz'
INDEX_HASH='69a1c34cb90dee8f6d0407b7c074e54328c82ef5b5c9c0f9c4572c74659c224b'
if not index_file.is_file() or hashlib.sha256(index_file.read_bytes()).hexdigest()!=INDEX_HASH:
    index_dir.mkdir(parents=True,exist_ok=True)
    temp=index_file.with_suffix('.building')
    with temp.open('wb') as output:
        for i in (1,2,3):
            with requests.get(BASE_URL+f'index/sciglass_four_properties_mol_398_1.npz.part{i}',stream=True,timeout=180) as r:
                r.raise_for_status()
                for chunk in r.iter_content(chunk_size=1048576):
                    if chunk:output.write(chunk)
    if hashlib.sha256(temp.read_bytes()).hexdigest()!=INDEX_HASH:
        temp.unlink(missing_ok=True)
        raise ValueError('Контрольная сумма индекса SciGlass не совпала')
    temp.replace(index_file)
print('✅ Индекс SciGlass готов')

catalog=ROOT/'GlassDesignML_Upgrade'/'SciGlass_Literature_Catalog_step397'/'run_20261110_185741_201214'/'source_catalog_step397.csv'
get_verified(BASE_URL+'source_catalog_step397.csv',catalog,'c7cbfcb84facb308adfc18d8bc7bb91a9945ceae4f9db3bc9e650959de115c34')
cache=ROOT/'GlassDesignML_Upgrade'/'SciGlass_DOI_Cache_step409'/'source_doi_cache_409.json'
get_verified(BASE_URL+'source_doi_cache_409.json',cache,'7f9c414f3f78c724a85e0250b8c66f0177f0304a70f79c132d205e798aa22465')
print('✅ Библиографический каталог и DOI-кэш готовы')

# Модели не хранятся в GitHub: они скачиваются с опубликованных автором ссылок.
MODEL_IDS={'density_extra_trees_final.joblib': '1a8FO6SmxLygKN-c2OrWJDKR-OjdxQXdu', 'refractive_index_extra_trees_final_v2.joblib': '17y7tKUddZHSYbecl344C3XFm6CTs9Vt-', 'rf_tg_final.joblib': '1Cn4dSABuMloTtLa15fGKUn21iONmIc7f', 'extra_trees_softening_final.joblib': '19CDRPCMfCYGUBaGhjLa6iJMBPlPPG_GM'}
import gdown
models_dir=ROOT/'GlassDesignML'/'models'
models_dir.mkdir(parents=True,exist_ok=True)
for filename,file_id in MODEL_IDS.items():
    path=models_dir/filename
    if not path.exists() or path.stat().st_size<1024*1024:
        gdown.download(id=file_id,output=str(path),quiet=False)
    if not path.exists() or path.stat().st_size<1024*1024:
        raise RuntimeError(f'Не удалось скачать модель {filename}. Проверьте доступность Google Drive.')
    print('✅ Модель:',filename,round(path.stat().st_size/1048576,1),'МиБ')
print('✅ Все файлы GlassDesign ML подготовлены')


In [ ]:
import time, re, subprocess, pathlib, requests
PROJECT=pathlib.Path('/content/glassdesign-ml')
port='8501'
app_log=open('/content/glassdesign_streamlit.log','w')
app=subprocess.Popen(['streamlit','run','streamlit_app.py','--server.port='+port,'--server.address=0.0.0.0','--server.headless=true','--browser.gatherUsageStats=false'],cwd=str(PROJECT),stdout=app_log,stderr=subprocess.STDOUT)
for _ in range(90):
    if app.poll() is not None:
        raise RuntimeError('Streamlit завершился. Откройте /content/glassdesign_streamlit.log')
    try:
        if requests.get('http://127.0.0.1:'+port+'/_stcore/health',timeout=1).ok:break
    except requests.RequestException:pass
    time.sleep(1)
else:
    raise RuntimeError('Streamlit не запустился за 90 секунд. Проверьте /content/glassdesign_streamlit.log')

cf=pathlib.Path('/content/cloudflared')
if not cf.exists():
    r=requests.get('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',timeout=120)
    r.raise_for_status()
    cf.write_bytes(r.content)
    cf.chmod(0o755)
tunnel_log=pathlib.Path('/content/glassdesign_tunnel.log')
with tunnel_log.open('w') as stream:
    tunnel=subprocess.Popen([str(cf),'tunnel','--no-autoupdate','--url','http://127.0.0.1:'+port],stdout=stream,stderr=subprocess.STDOUT)
url=None
for _ in range(90):
    if tunnel.poll() is not None:break
    time.sleep(1)
    match=re.search(r'https://[A-Za-z0-9-]+\.trycloudflare\.com',tunnel_log.read_text(errors='replace'))
    if match:
        url=match.group(0)
        break
if not url:
    raise RuntimeError('Не удалось получить ссылку Cloudflare. Откройте /content/glassdesign_tunnel.log')
print('\n✅ GLASSDESIGN ML ОТКРЫТ')
print('Ссылка на приложение:',url)
print('Оставьте среду Colab активной, пока пользуетесь сайтом.')
